
# DNTC strict OCR/PDF/NLP pipeline for Kaggle

Notebook này thay pipeline cũ bằng pipeline nghiêm ngặt hơn cho bộ scan **Đại Nam Nhất Thống Chí**:

- tự tải PDF từ Google Drive, hoặc đọc từ `/kaggle/input`;
- phân loại trang trước khi lấy text;
- chọn nguồn text theo từng trang: PDF text layer, Tesseract OCR, hoặc drop/audit;
- không đưa dòng/trang OCR thấp điểm vào final;
- sửa OCR tiếng Việt/domain theo luật có log;
- reflow paragraph, tách sentence-only;
- xuất final và audit đầy đủ vào `/kaggle/working/dntc_auto`.

Ưu tiên của notebook này là **precision hơn recall**: dòng nào OCR không chắc sẽ bị drop vào audit, không đưa vào final.


In [ ]:

# ============================================================
# 0. CONFIG - chỉnh ở đây rồi Run All
# ============================================================
from pathlib import Path
import os

# Google Drive folder/file URLs. Có thể thay bằng Drive folder của bạn.
DRIVE_URLS = [
    "https://drive.google.com/drive/folders/1QZzyaozPRLcm5Y2nmUUbigyVFnsX_tkW",
]

# Nếu không muốn download Drive, để DRIVE_URLS = [] và notebook đọc từ /kaggle/input.
ALLOW_KAGGLE_INPUT_FALLBACK = True
LOCAL_INPUT_DIRS = [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]

# Output đúng yêu cầu.
OUTPUT_DIR = Path("/kaggle/working/dntc_auto") if Path("/kaggle/working").exists() else Path("./dntc_auto")
RAW_DIR = OUTPUT_DIR / "raw_drive"
FINAL_DIR = OUTPUT_DIR / "final"
TEXT_DIR = FINAL_DIR / "texts"
AUDIT_DIR = OUTPUT_DIR / "audit"
CACHE_DIR = OUTPUT_DIR / "cache"
PKG_DIR = OUTPUT_DIR / "packages"

# Run modes.
FAST_TEST_MODE = False           # True: chạy thử nhanh một số trang.
FAST_TEST_MAX_PDFS = 2
FAST_TEST_MAX_PAGES_TOTAL = 50
FULL_RUN_MODE = not FAST_TEST_MODE
MAX_PAGES_PER_PDF = None         # None = all pages; FAST_TEST_MODE sẽ override bằng tổng 50 pages.

# Required page filtering config.
CONTENT_START_MODE = "auto"      # "auto" hoặc "none".
KEEP_TITLE_PAGES = False
DROP_LIBRARY_PAGES = True
DROP_BLANK_PAGES = True
DROP_PATTERNED_PAGES = True
DROP_LOW_CONF_OCR_PAGES = True
DROP_FRONT_MATTER_BEFORE_CONTENT = True

# OCR controls. Không dùng PaddleOCR mặc định vì dependency nặng/dễ lỗi trên Kaggle.
ENABLE_TESSERACT = True
OCR_LANG = "vie+eng"
PAGE_OCR_DPI = 260
FAST_FRONTMATTER_OCR_DPI = 150
LINE_REOCR_ZOOM = 4.0
LINE_REOCR_PAD_PT = 7.0
MAX_LINE_REOCR_PER_PDF = 250

# Quality thresholds. Tăng nếu muốn ít final hơn nhưng sạch hơn.
MIN_SELECTED_PAGE_QUALITY = 43.0
MIN_OCR_PAGE_QUALITY = 46.0
MIN_TEXT_LAYER_GOOD_QUALITY = 55.0
MIN_KEEP_LINE_QUALITY = 38.0
MIN_KEEP_SENTENCE_QUALITY = 38.0
MIN_OCR_CONF_LINE = 35.0
MAX_LIBRARY_NOISE_SCORE = 0.18
MAX_WEIRD_CHAR_RATIO = 0.065
MIN_VIET_RATIO_FOR_LONG_TEXT = 0.10

# Content detection.
AUTO_CONTENT_SCAN_PAGES = 35
MIN_CONTENT_WORDS_ON_START_PAGE = 35
MIN_CONTENT_LINES_ON_START_PAGE = 4

# Exports.
ZIP_OUTPUT = True
WRITE_DEBUG_PAGE_THUMBNAILS = False
DEBUG_THUMBNAIL_MAX = 80

for d in [OUTPUT_DIR, RAW_DIR, FINAL_DIR, TEXT_DIR, AUDIT_DIR, CACHE_DIR, PKG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("FAST_TEST_MODE:", FAST_TEST_MODE)
print("FULL_RUN_MODE:", FULL_RUN_MODE)


In [ ]:

# ============================================================
# 1. Install dependencies - Kaggle Run All friendly
# ============================================================
import sys, subprocess, shutil, importlib.util


def pip_install(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *pkgs]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)

required_modules = {
    "fitz": ["pymupdf"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "PIL": ["pillow"],
    "tqdm": ["tqdm"],
    "gdown": ["gdown"],
    "pytesseract": ["pytesseract"],
}
for mod, pkgs in required_modules.items():
    if importlib.util.find_spec(mod) is None:
        pip_install(*pkgs)

# Tesseract binary + Vietnamese and English traineddata.
if ENABLE_TESSERACT:
    missing_binary = shutil.which("tesseract") is None
    if missing_binary:
        print("Installing tesseract system packages...")
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie", "tesseract-ocr-eng"], check=False)
    else:
        # Ensure vie/eng packages exist. Safe if already installed.
        try:
            langs = subprocess.check_output(["tesseract", "--list-langs"], text=True, stderr=subprocess.STDOUT)
        except Exception:
            langs = ""
        if "vie" not in langs or "eng" not in langs:
            print("Installing tesseract Vietnamese/English traineddata...")
            subprocess.run(["apt-get", "update", "-qq"], check=False)
            subprocess.run(["apt-get", "install", "-y", "tesseract-ocr-vie", "tesseract-ocr-eng"], check=False)

print("tesseract:", shutil.which("tesseract"))


In [ ]:

# ============================================================
# 2. Imports and shared constants
# ============================================================
import os, re, math, json, time, zipfile, hashlib, shutil, subprocess, sys, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import fitz
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
import gdown

try:
    import pytesseract
    from pytesseract import Output
except Exception as e:
    pytesseract = None
    Output = None
    print("pytesseract not available:", repr(e))

VIET_CHARS = set(
    "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩị"
    "óòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ"
    "ĂÂĐÊÔƠƯÁÀẢÃẠẮẰẲẴẶẤẦẨẪẬÉÈẺẼẸẾỀỂỄỆÍÌỈĨỊ"
    "ÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ"
)
VIET_VOWELS = set("aeiouyAEIOUYàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵăâêôơưĂÂÊÔƠƯ")
LETTERS_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]")
WORDS_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+")
DIGIT_RE = re.compile(r"\d")
CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\ufffe\uffff\u0001]")
CJK_RE = re.compile(r"[\u3400-\u9fff]")
NON_VIET_SCRIPT_RE = re.compile(r"[Α-ωА-яЁё]")
WEIRD_RE = re.compile(r"[^\w\sÀ-ỹ.,;:!?(){}\[\]\"'“”‘’/\-–—%°+&<>«»·•*]", re.UNICODE)

COMMON_VI_WORDS = set("""
của và là có không trong ngoài năm đời phủ huyện tỉnh châu xã thôn làng tổng phường sách dân người nước ta
sông núi biển cửa đông tây nam bắc phía giáp cách dặm linh thành đặt đổi thuộc đời nhà triều vua quan quân
quyền quyển đại nam nhất thống chí kinh sư phần dã dựng đặt diễn cách hình thế khí hậu phong tục thành trì
trường học hộ khẩu thuế ruộng núi sông cổ tích từ miếu đền chùa miếu đàn lăng mộ đồn lũy cửa biển cầu đường
minh mệnh gia long tự đức thiệu trị đồng khánh duy tân hiến tông duệ tông thế tổ thánh tông cao hoàng đế
chiêm thành cao mên xiêm la thanh nghệ an thanh hóa quảng bình quảng trị quảng nam quảng ngãi bình định
phú yên khánh hòa bình thuận hà tiên an giang biên hòa gia định định tường vĩnh long hà nội hải dương
""".split())

COMMON_UNACCENTED_VI_WORDS = set("""
cua va la co khong trong ngoai nam doi phu huyen tinh chau xa thon lang tong phuong sach dan nguoi nuoc ta
song nui bien cua dong tay nam bac phia giap cach dam linh thanh dat doi thuoc nha trieu vua quan quyen quyen
dai nam nhat thong chi kinh su phan da dung dat dien cach hinh the khi hau phong tuc thanh tri truong hoc ho khau
thue ruong co tich tu mieu den chua minh menh gia long tu duc thieu tri dong khanh duy tan hien tong due tong
the to thanh tong chiem thanh cao men xiem la quang binh quang tri quang nam quang ngai binh dinh phu yen
khanh hoa binh thuan ha tien an giang bien hoa gia dinh dinh tuong vinh long ha noi hai duong
""".split())

DOMAIN_HEADINGS = [
    "ĐẠI NAM NHẤT THỐNG CHÍ", "LỜI NÓI ĐẦU", "BÀI TỰ", "PHÀM LỆ", "DỰNG ĐẶT VÀ DIÊN CÁCH", "PHẦN DÃ",
    "HÌNH THẾ", "KHÍ HẬU", "PHONG TỤC", "THÀNH TRÌ", "TRƯỜNG HỌC", "HỘ KHẨU", "THUẾ RUỘNG", "NÚI SÔNG",
    "SÔNG NGÒI", "CỔ TÍCH", "ĐỀN MIẾU", "TỪ MIẾU", "LĂNG MỘ", "ĐỒN LŨY", "CẦU ĐƯỜNG", "CHỢ QUÁN",
]

DOMAIN_TOKENS = set(" ".join(DOMAIN_HEADINGS).lower().split()) | COMMON_VI_WORDS

LIBRARY_NOISE_RE = re.compile(
    r"\b(library|libraries|university|barcode|digitized|google|riverside|michigan|wisconsin|madison|"
    r"east\s+west|center\s+library|memorial|archive|scan|call\s*number|state\s+street|copyright|"
    r"vol\.?|volume|DS\s*\d|G27|EAST WEST CENTER|THE UNIVERSITY)\b",
    re.I,
)
BARCODE_CALLNO_RE = re.compile(r"^(?:[A-Z]{0,3}\s*)?(?:DS|G|B|HV|VIET)?\s*[A-Z0-9.\-/ ]{2,18}$", re.I)
TITLE_FRONTMATTER_RE = re.compile(
    r"\b(văn\s*-?\s*hóa|tùng\s*-?\s*thư|dịch\s*-?\s*giả|soạn\s*-?\s*giả|xuất\s*-?\s*bản|"
    r"bộ\s+quốc\s*-?\s*gia|bộ\s+văn\s*-?\s*hóa|nhà\s+xuất\s+bản|thuận\s+hóa|tập\s+số|"
    r"tái\s+bản|người\s+dịch|người\s+hiệu\s+đính)\b",
    re.I,
)
CONTENT_START_RE = re.compile(
    r"\b(lời\s+nói\s+đầu|bài\s+tự|phàm\s+lệ|quyền\s+[ivxlcdm0-9]+|tỉnh\s+[A-ZÀ-ỸĐ]|"
    r"dựng\s+đặt|diên\s+cách|diễn\s+cách|phần\s+dã|hình\s+thế|phong\s+tục|đông\s+tây\s+cách\s+nhau)\b",
    re.I,
)

CLEAR_JUNK_RE = re.compile(
    r"(OPOC|erererore|Fel\s+Fat|Seer\s+tit|mADS|OKOK|OROR|RORO|Peete|Sarine|"
    r"^[\s:;,.`´‘’\"\\/\-–—_~|°*+={}\[\]()<>]{1,14}$|^\s*5\s*[-–—]\s*$)",
    re.I,
)
REPEATED_FRAGMENT_RE = re.compile(r"([A-Za-z]{2,5})\1{2,}")

print("imports ok")


In [ ]:

# ============================================================
# 3. Download / discover PDFs
# ============================================================

def download_drive_url(url: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    if not url or "PASTE" in url:
        return
    print("Downloading from Drive:", url)
    try:
        if "/folders/" in url:
            gdown.download_folder(url, output=str(out_dir), quiet=False, use_cookies=False, remaining_ok=True)
        else:
            gdown.download(url, output=str(out_dir), quiet=False, fuzzy=True)
    except TypeError:
        # Older gdown fallback.
        if "/folders/" in url:
            subprocess.run([sys.executable, "-m", "gdown", "--folder", url, "-O", str(out_dir)], check=False)
        else:
            subprocess.run([sys.executable, "-m", "gdown", url, "-O", str(out_dir)], check=False)
    except Exception as e:
        print("Drive download failed:", repr(e))

RAW_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_URLS:
    existing = list(RAW_DIR.rglob("*.pdf"))
    if not existing:
        for url in DRIVE_URLS:
            download_drive_url(url, RAW_DIR)

pdf_candidates = []
pdf_candidates += list(RAW_DIR.rglob("*.pdf"))
if ALLOW_KAGGLE_INPUT_FALLBACK:
    for d in LOCAL_INPUT_DIRS:
        if d.exists():
            pdf_candidates += list(d.rglob("*.pdf"))

seen = set()
pdf_paths = []
for p in sorted(pdf_candidates):
    try:
        key = (p.name.lower(), p.stat().st_size)
        if key in seen:
            continue
        seen.add(key)
        pdf_paths.append(p)
    except Exception:
        pass

if FAST_TEST_MODE:
    pdf_paths = pdf_paths[:FAST_TEST_MAX_PDFS]

if not pdf_paths:
    raise FileNotFoundError("No PDFs found. Check DRIVE_URLS, Internet=On, or /kaggle/input.")

print("PDF count:", len(pdf_paths))
for p in pdf_paths[:60]:
    print("-", p)
if len(pdf_paths) > 60:
    print("...", len(pdf_paths) - 60, "more")


In [ ]:

# ============================================================
# 4. Text normalization, metrics, quality score
# ============================================================

def norm_text(s) -> str:
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    replacements = {
        "\u00a0": " ", "\ufeff": " ", "￾": " ", "ﬁ": "fi", "ﬂ": "fl",
        "`": "'", "´": "'", "“": "\"", "”": "\"", "‘": "'", "’": "'",
    }
    for a, b in replacements.items():
        s = s.replace(a, b)
    s = CONTROL_RE.sub(" ", s)
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def normalize_for_match(s: str) -> str:
    s = norm_text(s).lower()
    s = re.sub(r"[\-–—_]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn").replace("đ", "d").replace("Đ", "D")


def count_repeated_fragment_ratio(s: str) -> float:
    s0 = re.sub(r"\s+", "", norm_text(s))
    if len(s0) < 8:
        return 0.0
    repeated_chars = sum(len(m.group(0)) for m in REPEATED_FRAGMENT_RE.finditer(s0))
    # Also catch long runs like eeeee or -----.
    repeated_chars += sum(len(m.group(0)) for m in re.finditer(r"(.)\1{4,}", s0))
    return min(1.0, repeated_chars / max(1, len(s0)))


def punctuation_balance(s: str) -> float:
    s = norm_text(s)
    pairs = [("(", ")"), ("[", "]"), ("{", "}"), ("«", "»"), ('"', '"')]
    imbalance = 0
    for a, b in pairs:
        if a == b:
            imbalance += abs(s.count(a) % 2)
        else:
            imbalance += abs(s.count(a) - s.count(b))
    punct_density = len(re.findall(r"[.,;:!?/\\|]{2,}", s))
    return min(1.0, (imbalance + punct_density) / max(1, len(s) / 40.0))


def library_noise_score(s: str) -> float:
    s0 = norm_text(s)
    if not s0:
        return 0.0
    hits = len(LIBRARY_NOISE_RE.findall(s0))
    callno = 1 if BARCODE_CALLNO_RE.fullmatch(s0) and not CONTENT_START_RE.search(s0) else 0
    digit_chunks = len(re.findall(r"\d{4,}", s0))
    score = 0.17 * hits + 0.12 * callno + 0.025 * digit_chunks
    return min(1.0, score)


def latin_noise_ratio(s: str) -> float:
    s0 = norm_text(s)
    words = WORDS_RE.findall(s0)
    if not words:
        return 0.0
    bad = 0
    for w in words:
        wl = w.lower()
        if wl in COMMON_VI_WORDS or wl in COMMON_UNACCENTED_VI_WORDS:
            continue
        if len(w) >= 4:
            vowel_count = sum(1 for ch in w if ch in VIET_VOWELS)
            if vowel_count == 0:
                bad += 1
            elif ord(max(w)) < 128 and len(w) >= 7 and vowel_count / len(w) < 0.22:
                bad += 1
    return bad / max(1, len(words))


def text_metrics(text: str, lines=None, ocr_conf_values=None) -> dict:
    s = norm_text(text)
    n = max(1, len(s))
    letters = LETTERS_RE.findall(s)
    words = [w.lower() for w in WORDS_RE.findall(s)]
    word_count = len(words)
    accent_count = sum(1 for ch in s if ch in VIET_CHARS)
    cjk_count = len(CJK_RE.findall(s))
    weird_count = len(WEIRD_RE.findall(s)) + 3 * len(NON_VIET_SCRIPT_RE.findall(s))
    controls = len(CONTROL_RE.findall(str(text or "")))
    digits = len(DIGIT_RE.findall(s))
    dictionary_hits = sum(1 for w in words if w in COMMON_VI_WORDS)
    unaccented_hits = sum(1 for w in words if w in COMMON_UNACCENTED_VI_WORDS)
    domain_hits = sum(1 for h in DOMAIN_HEADINGS if h.lower() in s.lower())
    vietnamese_ratio = min(1.0, (dictionary_hits + 0.45 * accent_count + 2.0 * domain_hits) / max(1, word_count))
    dictionary_hit_ratio = dictionary_hits / max(1, word_count)
    accent_ratio = accent_count / max(1, len(letters))
    weird_char_ratio = (weird_count + controls) / n
    digit_ratio = digits / n
    rep_ratio = count_repeated_fragment_ratio(s)
    lib_score = library_noise_score(s)
    lat_noise = latin_noise_ratio(s)
    avg_line_length = 0.0
    if lines:
        nonempty = [norm_text(x) for x in lines if norm_text(x)]
        avg_line_length = float(np.mean([len(x) for x in nonempty])) if nonempty else 0.0
    else:
        avg_line_length = len(s)
    punct_bal = punctuation_balance(s)
    avg_ocr_conf = None
    if ocr_conf_values:
        vals = [float(x) for x in ocr_conf_values if x is not None and not pd.isna(x) and float(x) >= 0]
        avg_ocr_conf = float(np.mean(vals)) if vals else None

    # Quality score 0-100-ish. Designed to be conservative.
    score = 50.0
    score += 24.0 * dictionary_hit_ratio
    score += 18.0 * vietnamese_ratio
    score += min(8.0, avg_line_length / 12.0)
    if word_count >= 20:
        score += 4.0
    if accent_ratio >= 0.04:
        score += 4.0
    score -= 95.0 * weird_char_ratio
    score -= 28.0 * lat_noise
    score -= 30.0 * rep_ratio
    score -= 24.0 * lib_score
    score -= 12.0 * punct_bal
    score -= 16.0 * digit_ratio if digit_ratio > 0.22 else 0.0
    if word_count < 4 and not domain_hits:
        score -= 28.0
    if len(letters) < 15 and not domain_hits:
        score -= 18.0
    if word_count >= 12 and accent_ratio < 0.018 and unaccented_hits < 2 and dictionary_hits < 2:
        score -= 14.0
    if avg_ocr_conf is not None:
        if avg_ocr_conf < 25:
            score -= 16.0
        elif avg_ocr_conf < 38:
            score -= 8.0
        elif avg_ocr_conf > 55:
            score += 3.0
    if CLEAR_JUNK_RE.search(s):
        score -= 32.0
    if CONTENT_START_RE.search(s):
        score += 5.0
    return {
        "char_count": len(s), "letter_count": len(letters), "word_count": word_count,
        "accent_count": accent_count, "accent_ratio": accent_ratio, "cjk_count": cjk_count,
        "weird_char_count": weird_count, "control_count": controls, "weird_char_ratio": weird_char_ratio,
        "vietnamese_ratio": vietnamese_ratio, "dictionary_hit_ratio": dictionary_hit_ratio,
        "unaccented_vi_hits": unaccented_hits, "domain_heading_hits": domain_hits,
        "latin_noise_ratio": lat_noise, "avg_line_length": avg_line_length,
        "punctuation_balance": punct_bal, "digit_ratio": digit_ratio,
        "repeated_char_ngram_ratio": rep_ratio, "library_noise_score": lib_score,
        "avg_ocr_conf": avg_ocr_conf, "quality_score": round(score, 3),
    }


def is_heading_text(s: str) -> bool:
    s0 = norm_text(s)
    if not s0 or len(s0) > 110:
        return False
    su = s0.upper()
    if any(h in su for h in DOMAIN_HEADINGS):
        return True
    if re.fullmatch(r"(?:QUYỂN|QUYEN|TẬP|TAP|TỈNH|TINH)\s+[A-ZÀ-ỸĐ0-9IVXLCDM .\-]+", su):
        return True
    letters = LETTERS_RE.findall(s0)
    if len(letters) >= 4:
        upperish = sum(1 for ch in letters if ch.upper() == ch) / len(letters)
        if upperish >= 0.82 and len(s0) <= 80 and not LIBRARY_NOISE_RE.search(s0):
            return True
    return False


def is_title_or_frontmatter_text(s: str, page_number: int) -> bool:
    s0 = norm_text(s)
    if not s0:
        return False
    if page_number <= AUTO_CONTENT_SCAN_PAGES and TITLE_FRONTMATTER_RE.search(s0):
        # A long actual preface page can mention publisher; do not classify as title if content terms dominate.
        words = WORDS_RE.findall(s0)
        if len(words) < 70 or not CONTENT_START_RE.search(s0):
            return True
    title_only_tokens = ["ĐẠI NAM", "NHẤT THỐNG", "DỊCH GIẢ", "XUẤT BẢN", "NHÀ XUẤT BẢN", "TẬP SỐ"]
    hit = sum(1 for t in title_only_tokens if t.lower() in s0.lower())
    if page_number <= AUTO_CONTENT_SCAN_PAGES and hit >= 2 and len(WORDS_RE.findall(s0)) < 80:
        return True
    return False


In [ ]:

# ============================================================
# 5. Rendering, visual page metrics, OCR, PDF text extraction
# ============================================================

def tesseract_available() -> bool:
    return ENABLE_TESSERACT and pytesseract is not None and shutil.which("tesseract") is not None


def render_page_to_pil(page, dpi=90, clip=None) -> Image.Image:
    zoom = dpi / 72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), clip=clip, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def visual_page_metrics(page) -> dict:
    try:
        img = render_page_to_pil(page, dpi=45)
        arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
        gray = arr.mean(axis=2)
        whiteness = float(np.mean(gray > 0.94))
        darkness = float(np.mean(gray < 0.12))
        ink_ratio = float(np.mean(gray < 0.88))
        color_std = float(np.mean(np.std(arr, axis=(0, 1))))
        mx = arr.max(axis=2)
        mn = arr.min(axis=2)
        saturation = float(np.mean((mx - mn) / np.maximum(mx, 1e-6)))
        gy = np.abs(np.diff(gray, axis=0)).mean()
        gx = np.abs(np.diff(gray, axis=1)).mean()
        edge_density = float(gx + gy)
        h, w = gray.shape
        border = max(2, int(min(h, w) * 0.035))
        border_pixels = np.concatenate([
            gray[:border, :].ravel(), gray[-border:, :].ravel(), gray[:, :border].ravel(), gray[:, -border:].ravel()
        ])
        border_ink_ratio = float(np.mean(border_pixels < 0.80))
        return {
            "white_ratio": whiteness, "dark_ratio": darkness, "ink_ratio": ink_ratio,
            "color_std": color_std, "saturation": saturation, "edge_density": edge_density,
            "border_ink_ratio": border_ink_ratio,
            "visual_blank_score": float(whiteness > 0.985 and ink_ratio < 0.018),
            "visual_pattern_score": min(1.0, max(0.0, saturation * 1.8 + color_std * 1.2 + edge_density * 4.0 - whiteness * 0.75)),
        }
    except Exception as e:
        return {"visual_error": type(e).__name__, "white_ratio": None, "dark_ratio": None, "ink_ratio": None,
                "color_std": None, "saturation": None, "edge_density": None, "border_ink_ratio": None,
                "visual_blank_score": 0.0, "visual_pattern_score": 0.0}


def pil_preprocess_for_ocr(img: Image.Image, mode="page") -> Image.Image:
    img = img.convert("RGB")
    gray = ImageOps.grayscale(img)
    gray = ImageOps.autocontrast(gray)
    gray = ImageEnhance.Contrast(gray).enhance(1.35 if mode == "page" else 1.65)
    gray = ImageEnhance.Sharpness(gray).enhance(1.20 if mode == "page" else 1.55)
    if mode == "line":
        gray = gray.filter(ImageFilter.MedianFilter(size=3))
    return gray


def tesseract_data_from_image(img: Image.Image, psm=6):
    if not tesseract_available():
        return pd.DataFrame()
    try:
        config = f"--oem 1 --psm {psm}"
        df = pytesseract.image_to_data(img, lang=OCR_LANG, config=config, output_type=Output.DATAFRAME)
        if df is None:
            return pd.DataFrame()
        return df
    except Exception as e:
        return pd.DataFrame()


def extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=PAGE_OCR_DPI) -> list:
    if not tesseract_available():
        return []
    img = render_page_to_pil(page, dpi=dpi)
    zoom = dpi / 72.0
    img = pil_preprocess_for_ocr(img, mode="page")
    df = tesseract_data_from_image(img, psm=6)
    if df.empty:
        return []
    df = df.dropna(subset=["text"]).copy()
    if df.empty:
        return []
    df["text"] = df["text"].map(norm_text)
    df = df[df["text"].str.len() > 0]
    if df.empty:
        return []
    if "conf" in df.columns:
        df["conf_num"] = pd.to_numeric(df["conf"], errors="coerce").fillna(-1)
    else:
        df["conf_num"] = -1
    group_cols = [c for c in ["block_num", "par_num", "line_num"] if c in df.columns]
    if not group_cols:
        group_cols = ["level"] if "level" in df.columns else []
    if not group_cols:
        return []
    lines = []
    for _, g in df.groupby(group_cols, sort=True):
        words = [norm_text(x) for x in g["text"].tolist() if norm_text(x)]
        text = norm_text(" ".join(words))
        if not text:
            continue
        conf_vals = [float(x) for x in g["conf_num"].tolist() if float(x) >= 0]
        conf = float(np.mean(conf_vals)) if conf_vals else -1.0
        left = float(g["left"].min()) / zoom
        top = float(g["top"].min()) / zoom
        right = float((g["left"] + g["width"]).max()) / zoom
        bottom = float((g["top"] + g["height"]).max()) / zoom
        lines.append({
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_idx + 1,
            "source": "tesseract_page", "bbox": [left, top, right, bottom], "raw_text": text,
            "ocr_conf": round(conf, 3), "block_id": None, "line_id": None,
        })
    lines.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return lines


def extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path) -> list:
    out = []
    try:
        data = page.get_text("dict")
    except Exception:
        return out
    for b_i, block in enumerate(data.get("blocks", [])):
        if block.get("type") != 0:
            continue
        for l_i, line in enumerate(block.get("lines", [])):
            spans = line.get("spans", [])
            text = norm_text(" ".join([sp.get("text", "") for sp in spans]))
            if not text:
                continue
            bbox = line.get("bbox", block.get("bbox", None))
            if not bbox:
                continue
            font_sizes = [float(sp.get("size", 0) or 0) for sp in spans]
            out.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_idx + 1,
                "source": "pdf_text_layer", "bbox": [float(x) for x in bbox], "raw_text": text,
                "ocr_conf": None, "block_id": b_i, "line_id": l_i,
                "font_size_avg": round(float(np.mean(font_sizes)) if font_sizes else 0.0, 3),
            })
    out.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return out


def lines_to_text(lines) -> str:
    return "\n".join(norm_text(r.get("raw_text") or r.get("text")) for r in lines if norm_text(r.get("raw_text") or r.get("text")))


def line_crop_ocr(page, bbox) -> tuple:
    if not tesseract_available():
        return "", -1.0
    r = fitz.Rect(bbox)
    r.x0 = max(page.rect.x0, r.x0 - LINE_REOCR_PAD_PT)
    r.y0 = max(page.rect.y0, r.y0 - LINE_REOCR_PAD_PT)
    r.x1 = min(page.rect.x1, r.x1 + LINE_REOCR_PAD_PT)
    r.y1 = min(page.rect.y1, r.y1 + LINE_REOCR_PAD_PT)
    pix = page.get_pixmap(matrix=fitz.Matrix(LINE_REOCR_ZOOM, LINE_REOCR_ZOOM), clip=r, alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    variants = [pil_preprocess_for_ocr(img, mode="line"), img]
    best_text, best_conf, best_score = "", -1.0, -999.0
    for im in variants:
        df = tesseract_data_from_image(im, psm=7)
        if df.empty:
            continue
        df = df.dropna(subset=["text"]).copy()
        if df.empty:
            continue
        txt = norm_text(" ".join(df["text"].map(norm_text).tolist()))
        conf = -1.0
        if "conf" in df.columns:
            vals = pd.to_numeric(df["conf"], errors="coerce")
            vals = vals[vals >= 0]
            if len(vals):
                conf = float(vals.mean())
        score = text_metrics(txt, ocr_conf_values=[conf]).get("quality_score", -999.0)
        if score > best_score:
            best_text, best_conf, best_score = txt, conf, score
    return best_text, best_conf


In [ ]:

# ============================================================
# 6. Conservative Vietnamese/domain correction with log
# ============================================================
CORRECTION_RULES = [
    # Headings and title variants.
    ("heading_dai_nam", r"\bDAI\s*-?\s*NAM\b|\bĐAI\s*-?\s*NAM\b", "ĐẠI NAM"),
    ("heading_nhat_thong_chi", r"\bNH[ẤA]T\s*-?\s*TH[OỐ]NG\s*-?\s*CH[ÍI]\b|\bNHAT\s+THONG\s+CHI\b", "NHẤT THỐNG CHÍ"),
    ("heading_quyen", r"\bQUYEN\b", "QUYỂN"),
    ("heading_tinh", r"\bTINH\b(?=\s+[A-ZÀ-ỸĐ])", "TỈNH"),
    ("heading_phan_da", r"\bPHAN\s+DA\b|\bPHẦN\s+DA\b", "PHẦN DÃ"),
    ("heading_dung_dat", r"\bD[UƯ]NG\s+[DPĐ]AT\s+VA\s+DI[ÊE]N\s+C[ÁA]CH\b|\bDUNG\s+DAT\s+VA\s+DIEN\s+CACH\b", "DỰNG ĐẶT VÀ DIÊN CÁCH"),
    ("heading_hinh_the", r"\bHINH\s+THE\b", "HÌNH THẾ"),
    ("heading_phong_tuc", r"\bPHONG\s+TUC\b", "PHONG TỤC"),
    ("heading_thue_ruong", r"\bTHUE\s+RUONG\b", "THUẾ RUỘNG"),
    ("heading_nui_song", r"\bNUI\s+SONG\b", "NÚI SÔNG"),

    # Specific OCR/mojibake artifacts reported for DNTC.
    ("mojibake_chiem", r"\bchi€m\b", "chiếm"),
    ("mojibake_bien", r"\bbi€n\b", "biển"),
    ("mojibake_kiem", r"\bki€m\b", "kiêm"),
    ("mojibake_mieu", r"\bmi€u\b", "miếu"),
    ("n_tilde_nam", r"\bñăm\b", "năm"),
    ("the_ky", r"\bth[eéế]\s+k[yỷ]\b|\bth[eéế]\s+kỷ\b", "thế kỷ"),
    ("dau_the_ky", r"\bĐầu\s+th[eéế]\s+k[yỷ]\b", "Đầu thế kỷ"),
    ("doi_tu_duc", r"\bdoi\s+#?ự\s+Đức\b|\bđời\s+#?ự\s+Đức\b", "đời Tự Đức"),
    ("doi_hien_tong", r"\bdoi\s+H[ií]én\s+Tông\b", "đời Hiến Tông"),
    ("nha_tuy", r"\bNha\s+Tuy\b", "Nhà Tùy"),
    ("nha_duong", r"\bNha\s+Đường\b", "Nhà Đường"),
    ("nuoc_ta", r"\bNước\s+tả\b", "Nước ta"),
    ("cao_men", r"\bCao\s+M[eé]n\b", "Cao Mên"),
    ("huyen_hien", r"\bhuyénhién\b", "huyện hiện"),
    ("cuop", r"\bcudp\b", "cướp"),

    # Contextual administrative words. Conservative: only in common administrative contexts.
    ("tinh_ly", r"\btinh\s+l[yỵi]\b", "tỉnh lỵ"),
    ("tinh_thanh", r"\btinh\s+thành\b", "tỉnh thành"),
    ("tinh_place", r"\btinh\s+(Quảng|Thanh|Nghệ|Bình|Phú|Khánh|Hà|An|Gia|Định|Vĩnh|Biên|Quy|Hải|Nam|Bắc)\b", r"tỉnh \1"),
    ("huyen_context", r"\bhuyen\s+(?=[A-ZÀ-ỸĐ])", "huyện "),
    ("phu_context", r"\bphu\s+(?=[A-ZÀ-ỸĐ])", "phủ "),
    ("chau_context", r"\bchau\s+(?=[A-ZÀ-ỸĐ])", "châu "),

    # Common names.
    ("minh_menh", r"\bMinh\s+M[ée]nh\b|\bMinh\s+Mộệnh\b", "Minh Mệnh"),
    ("thieu_tri", r"\bThiệu\s+Tri\b", "Thiệu Trị"),
    ("tu_duc", r"\bTự\s+Dức\b|\bDự\s+Đức\b", "Tự Đức"),
    ("gia_du", r"\bGia\s+Du\s+Hoàng\b", "Gia Dụ Hoàng"),
    ("ha_tien", r"\bHa\s+Tien\b", "Hà Tiên"),
    ("quang_binh", r"\bQuang\s+Binh\b", "Quảng Bình"),
    ("binh_dinh", r"\bBinh\s+Dinh\b", "Bình Định"),
    ("quang_yen", r"\bQuang\s+Yen\b", "Quảng Yên"),
]


def apply_corrections(text: str, meta: dict, correction_log: list) -> str:
    original = norm_text(text)
    s = original
    # First normalize punctuation and known glyphs.
    pre = s
    s = s.replace("￾", " ").replace("\u0001", " ")
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,.;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(\d)\s+([,.])\s+(\d)", r"\1\2\3", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s != pre:
        correction_log.append({**meta, "level": "normalize", "rule_id": "unicode_space_punct", "before": pre, "after": s})
    for rule_id, pat, repl in CORRECTION_RULES:
        before = s
        s, n = re.subn(pat, repl, s, flags=re.IGNORECASE)
        if n and s != before:
            correction_log.append({**meta, "level": "domain_rule", "rule_id": rule_id, "before": before, "after": s})
    s = re.sub(r"\s+", " ", s).strip()
    return s


def candidate_correction_only(text: str) -> str:
    tmp_log = []
    return apply_corrections(text, {}, tmp_log)


def choose_better_line_text(base_text: str, ocr_text: str, ocr_conf: float, meta: dict, correction_log: list) -> tuple:
    base_fixed = apply_corrections(base_text, meta, correction_log)
    ocr_fixed = candidate_correction_only(ocr_text)
    if not ocr_fixed:
        return base_fixed, "rules_only", False, ""
    base_m = text_metrics(base_fixed)
    ocr_m = text_metrics(ocr_fixed, ocr_conf_values=[ocr_conf])
    # Keep base if OCR loses too many digits or becomes much shorter.
    base_digits = re.findall(r"\d+(?:[.,]\d+)?", base_fixed)
    if base_digits:
        kept = sum(1 for d in base_digits if d in ocr_fixed)
        if kept / max(1, len(base_digits)) < 0.70:
            return base_fixed, "keep_base_ocr_lost_digits", False, ocr_fixed
    if len(ocr_fixed) < 0.55 * max(1, len(base_fixed)):
        return base_fixed, "keep_base_ocr_too_short", False, ocr_fixed
    if ocr_conf is not None and ocr_conf >= 0 and ocr_conf < 28 and ocr_m["quality_score"] < base_m["quality_score"] + 10:
        return base_fixed, "keep_base_low_ocr_conf", False, ocr_fixed
    if ocr_m["quality_score"] >= base_m["quality_score"] + 8 and ocr_m["weird_char_ratio"] <= base_m["weird_char_ratio"] + 0.02:
        # Log accepted line OCR as a correction event.
        correction_log.append({**meta, "level": "line_ocr", "rule_id": "accepted_better_line_ocr", "before": base_fixed, "after": ocr_fixed})
        return ocr_fixed, f"accept_line_ocr {base_m['quality_score']:.1f}->{ocr_m['quality_score']:.1f} conf={ocr_conf:.1f}", True, ocr_fixed
    return base_fixed, f"keep_base {base_m['quality_score']:.1f}->{ocr_m['quality_score']:.1f} conf={ocr_conf:.1f}", False, ocr_fixed


In [ ]:

# ============================================================
# 7. Page classification and source selection
# ============================================================

def is_blank_page(text_m: dict, visual_m: dict) -> bool:
    if text_m["word_count"] <= 2 and (visual_m.get("visual_blank_score") == 1.0 or (visual_m.get("white_ratio") or 0) > 0.975):
        return True
    if text_m["letter_count"] < 5 and (visual_m.get("ink_ratio") or 1) < 0.025:
        return True
    return False


def is_patterned_page(text_m: dict, visual_m: dict, page_number: int) -> bool:
    if page_number > AUTO_CONTENT_SCAN_PAGES and text_m["word_count"] > 12:
        return False
    pattern_visual = (visual_m.get("visual_pattern_score") or 0) > 0.38 and (visual_m.get("white_ratio") or 1) < 0.82
    bad_text = text_m["vietnamese_ratio"] < 0.12 and text_m["word_count"] < 45
    return bool(pattern_visual and bad_text)


def is_library_page(text: str, text_m: dict) -> bool:
    if LIBRARY_NOISE_RE.search(norm_text(text)):
        return True
    if text_m["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE and text_m["vietnamese_ratio"] < 0.25:
        return True
    return False


def page_content_signal(text: str, text_m: dict) -> float:
    s = norm_text(text)
    score = 0.0
    score += min(35.0, text_m["word_count"] * 0.45)
    score += 25.0 * text_m["vietnamese_ratio"]
    score += 18.0 if CONTENT_START_RE.search(s) else 0.0
    score += 10.0 if any(h.lower() in s.lower() for h in DOMAIN_HEADINGS) else 0.0
    score -= 22.0 if TITLE_FRONTMATTER_RE.search(s) and text_m["word_count"] < 70 else 0.0
    score -= 35.0 * text_m["library_noise_score"]
    score -= 18.0 * text_m["repeated_char_ngram_ratio"]
    return score


def classify_page(page_number: int, text: str, text_m: dict, visual_m: dict, content_start_page: int | None) -> tuple:
    reasons = []
    cls = "content_candidate"
    if DROP_BLANK_PAGES and is_blank_page(text_m, visual_m):
        return "blank_page", ["blank_visual_or_no_text"]
    if DROP_PATTERNED_PAGES and is_patterned_page(text_m, visual_m, page_number):
        return "patterned_endpaper", ["patterned_visual_low_text"]
    if DROP_LIBRARY_PAGES and is_library_page(text, text_m):
        return "library_barcode_stamp_watermark", ["library_barcode_google_tokens"]
    if content_start_page is not None and DROP_FRONT_MATTER_BEFORE_CONTENT and page_number < content_start_page:
        return "front_matter_before_content", [f"before_auto_content_start_{content_start_page}"]
    if not KEEP_TITLE_PAGES and is_title_or_frontmatter_text(text, page_number):
        return "cover_title_front_matter", ["title_or_publisher_page"]
    if text_m["quality_score"] < 25 and text_m["word_count"] < 15:
        return "junk_ocr_page", ["low_text_quality_too_few_words"]
    return cls, reasons


def estimate_content_start(doc, work_id, pdf_path) -> int:
    if CONTENT_START_MODE != "auto":
        return 1
    max_scan = min(len(doc), AUTO_CONTENT_SCAN_PAGES)
    candidates = []
    for page_idx in range(max_scan):
        page = doc[page_idx]
        # Use text layer first. If text layer is nearly empty/bad, do a cheap OCR pass for content-start detection only.
        tl_lines = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
        tl_text = lines_to_text(tl_lines)
        tl_m = text_metrics(tl_text, lines=[r.get("raw_text", "") for r in tl_lines])
        detection_text = tl_text
        detection_m = tl_m
        if tl_m["letter_count"] < 20 or tl_m["weird_char_ratio"] > 0.10:
            try:
                ocr_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=FAST_FRONTMATTER_OCR_DPI)
                ocr_text = lines_to_text(ocr_lines)
                ocr_m = text_metrics(ocr_text, lines=[r.get("raw_text", "") for r in ocr_lines], ocr_conf_values=[r.get("ocr_conf") for r in ocr_lines])
                if ocr_m["quality_score"] > detection_m["quality_score"]:
                    detection_text, detection_m = ocr_text, ocr_m
            except Exception:
                pass
        visual_m = visual_page_metrics(page)
        preliminary_cls, _ = classify_page(page_idx + 1, detection_text, detection_m, visual_m, content_start_page=None)
        signal = page_content_signal(detection_text, detection_m)
        has_start = CONTENT_START_RE.search(norm_text(detection_text)) is not None
        title_front = is_title_or_frontmatter_text(detection_text, page_idx + 1)
        candidates.append({
            "page_number": page_idx + 1, "signal": signal, "has_start": has_start,
            "word_count": detection_m["word_count"], "line_count": len(detection_text.splitlines()),
            "quality_score": detection_m["quality_score"], "preliminary_cls": preliminary_cls,
            "title_front": title_front,
        })
    # Prefer the first page that has actual prose/content, not title-only.
    for c in candidates:
        if c["preliminary_cls"] in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark"}:
            continue
        if c["title_front"] and not KEEP_TITLE_PAGES:
            continue
        if c["has_start"] and c["word_count"] >= MIN_CONTENT_WORDS_ON_START_PAGE and c["quality_score"] >= 30:
            return int(c["page_number"])
    for c in candidates:
        if c["preliminary_cls"] == "content_candidate" and c["signal"] >= 35 and c["word_count"] >= MIN_CONTENT_WORDS_ON_START_PAGE:
            return int(c["page_number"])
    # Safe fallback: first page after obvious front matter with decent words.
    for c in candidates:
        if c["preliminary_cls"] == "content_candidate" and not c["title_front"] and c["word_count"] >= 25:
            return int(c["page_number"])
    return 1


def select_page_source(page, page_idx, work_id, pdf_path, page_class) -> tuple:
    # Always inspect text layer. OCR only if needed.
    tl_lines = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
    tl_text = lines_to_text(tl_lines)
    tl_m = text_metrics(tl_text, lines=[r.get("raw_text", "") for r in tl_lines])

    should_ocr = ENABLE_TESSERACT and tesseract_available() and (
        tl_m["quality_score"] < MIN_TEXT_LAYER_GOOD_QUALITY or
        tl_m["letter_count"] < 40 or
        tl_m["weird_char_ratio"] > 0.06 or
        tl_m["repeated_char_ngram_ratio"] > 0.08
    )
    ocr_lines, ocr_text, ocr_m = [], "", None
    if should_ocr and page_class not in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark", "cover_title_front_matter", "front_matter_before_content"}:
        ocr_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=PAGE_OCR_DPI)
        ocr_text = lines_to_text(ocr_lines)
        ocr_m = text_metrics(ocr_text, lines=[r.get("raw_text", "") for r in ocr_lines], ocr_conf_values=[r.get("ocr_conf") for r in ocr_lines])

    # Source decision.
    if ocr_m is None or not ocr_lines:
        selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
        reason = "text_layer_only_or_ocr_unavailable"
    else:
        # Do not let OCR replace a good text layer unless it is clearly better.
        if tl_m["quality_score"] >= MIN_TEXT_LAYER_GOOD_QUALITY and tl_m["quality_score"] >= ocr_m["quality_score"] - 7:
            selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
            reason = f"keep_text_layer_good tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        elif ocr_m["quality_score"] >= tl_m["quality_score"] + 7 and ocr_m["quality_score"] >= MIN_OCR_PAGE_QUALITY:
            selected, source, selected_m = ocr_lines, "tesseract_page", ocr_m
            reason = f"use_ocr_better tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        elif tl_m["quality_score"] >= 35 and tl_m["word_count"] >= 8:
            selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
            reason = f"fallback_text_layer_ocr_not_good_enough tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        else:
            selected, source, selected_m = [], "drop_no_reliable_source", max([tl_m, ocr_m], key=lambda x: x["quality_score"])
            reason = f"drop_both_sources_low tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
    return selected, source, selected_m, tl_m, (ocr_m or {}), reason


In [ ]:

# ============================================================
# 8. Line filtering, paragraph reflow, sentence splitting
# ============================================================

def line_quality(text: str, ocr_conf=None) -> dict:
    return text_metrics(text, lines=[text], ocr_conf_values=[ocr_conf] if ocr_conf is not None else None)


def is_repeated_junk_line(s: str) -> bool:
    s0 = norm_text(s)
    if not s0:
        return True
    if CLEAR_JUNK_RE.search(s0):
        return True
    if REPEATED_FRAGMENT_RE.search(re.sub(r"\s+", "", s0)):
        return True
    words = WORDS_RE.findall(s0)
    if words and len(words) <= 5:
        no_vowels = sum(1 for w in words if len(w) >= 3 and not any(ch in VIET_VOWELS for ch in w))
        if no_vowels / max(1, len(words)) > 0.55:
            return True
    # Patterned endpapers often OCR as B, ER, 83, 3, ॐ repeated.
    if re.fullmatch(r"[\sBЕER83()*0-9ॐఓజిஆ६]+", s0, flags=re.I):
        return True
    return False


def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    meta = {
        "work_id": line.get("work_id"), "pdf_path": line.get("pdf_path"), "page_number": line.get("page_number"),
        "page_idx": line.get("page_idx"), "line_id": line.get("line_global_id"), "source": line.get("source"),
    }
    reasons = []
    if not raw:
        return None, False, ["empty"]
    if LIBRARY_NOISE_RE.search(raw):
        return None, False, ["library_barcode_google_line"]
    if re.fullmatch(r"\d{1,4}", raw):
        return None, False, ["page_number_only"]
    if BARCODE_CALLNO_RE.fullmatch(raw) and len(WORDS_RE.findall(raw)) <= 3 and not is_heading_text(raw):
        return None, False, ["call_number_or_barcode_fragment"]
    if is_repeated_junk_line(raw):
        return None, False, ["repeated_or_known_ocr_junk"]

    fixed = apply_corrections(raw, meta, correction_log)
    # Optional line re-OCR only for suspicious PDF text layer lines. Do not replace OCR-page lines with another OCR.
    line_ocr_text = ""
    line_ocr_conf = None
    line_ocr_accepted = False
    if (doc is not None and reocr_state is not None and line.get("source") == "pdf_text_layer" and
        reocr_state.get("used", 0) < MAX_LINE_REOCR_PER_PDF):
        q0 = line_quality(fixed)
        suspicious_for_reocr = (
            q0["quality_score"] < MIN_KEEP_LINE_QUALITY + 8 or q0["weird_char_ratio"] > 0.035 or
            q0["repeated_char_ngram_ratio"] > 0.04 or q0["unaccented_vi_hits"] >= 2 and q0["accent_ratio"] < 0.015
        )
        if suspicious_for_reocr:
            try:
                page = doc[line["page_idx"]]
                line_ocr_text, line_ocr_conf = line_crop_ocr(page, line["bbox"])
                reocr_state["used"] = reocr_state.get("used", 0) + 1
                fixed2, reason, accepted, line_ocr_text2 = choose_better_line_text(fixed, line_ocr_text, line_ocr_conf, meta, correction_log)
                if accepted:
                    fixed = fixed2
                    line_ocr_accepted = True
                reasons.append(reason)
            except Exception as e:
                reasons.append(f"line_reocr_failed:{type(e).__name__}")

    q = line_quality(fixed, line.get("ocr_conf"))
    heading = is_heading_text(fixed)
    if not heading:
        if len(fixed) < 7 or q["letter_count"] < 3:
            return None, False, reasons + ["too_short_not_heading"]
        if q["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE:
            return None, False, reasons + ["library_noise_score"]
        if q["weird_char_ratio"] > MAX_WEIRD_CHAR_RATIO:
            return None, False, reasons + ["too_many_weird_chars"]
        if q["repeated_char_ngram_ratio"] > 0.12:
            return None, False, reasons + ["repeated_ngram_ratio"]
        if q["word_count"] >= 7 and q["vietnamese_ratio"] < MIN_VIET_RATIO_FOR_LONG_TEXT and q["accent_ratio"] < 0.025:
            return None, False, reasons + ["low_vietnamese_ratio"]
        if line.get("source") == "tesseract_page" and line.get("ocr_conf") is not None and float(line.get("ocr_conf")) >= 0:
            if float(line.get("ocr_conf")) < MIN_OCR_CONF_LINE and q["quality_score"] < MIN_KEEP_LINE_QUALITY + 8:
                return None, False, reasons + ["low_tesseract_conf"]
        if q["quality_score"] < MIN_KEEP_LINE_QUALITY:
            return None, False, reasons + [f"low_line_quality:{q['quality_score']:.1f}"]

    kept = dict(line)
    kept["text"] = fixed
    kept["line_type"] = "heading" if heading else "body"
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    kept["line_ocr_text"] = line_ocr_text
    kept["line_ocr_conf"] = line_ocr_conf
    kept["line_ocr_accepted"] = line_ocr_accepted
    kept["filter_reasons"] = ";".join(reasons) if reasons else "kept"
    return kept, True, reasons


def median_line_height(lines) -> float:
    vals = []
    for r in lines:
        try:
            b = r["bbox"]
            vals.append(max(1.0, float(b[3]) - float(b[1])))
        except Exception:
            pass
    return float(np.median(vals)) if vals else 12.0


def is_footnote_line(line: dict, page_rect) -> bool:
    try:
        y0 = line["bbox"][1]
        fs = line.get("font_size_avg") or 0
        text = line.get("text", "")
        return (y0 > page_rect.height * 0.80 and (re.match(r"^\(?\d+\)|^\*", text) or (fs and fs < 9)))
    except Exception:
        return False


def should_new_paragraph(prev: dict, cur: dict, med_h: float, page_rect) -> bool:
    pt, ct = prev.get("text", ""), cur.get("text", "")
    if prev.get("line_type") == "heading" or cur.get("line_type") == "heading":
        return True
    if prev.get("is_footnote") != cur.get("is_footnote"):
        return True
    try:
        gap = cur["bbox"][1] - prev["bbox"][3]
        indent_delta = cur["bbox"][0] - prev["bbox"][0]
        if gap > med_h * 0.95:
            return True
        if indent_delta > med_h * 1.7 and len(pt) > 25:
            return True
    except Exception:
        pass
    if re.match(r"^\(?\d+\)|^\d+[.)]", ct):
        return True
    return False


def join_paragraph_lines(lines: list) -> str:
    parts = []
    for r in lines:
        t = norm_text(r.get("text", ""))
        if not t:
            continue
        if not parts:
            parts.append(t)
            continue
        prev = parts[-1]
        # Remove true line-break hyphen only for lowercase/letter split words.
        if re.search(r"[a-zà-ỹđ]-$", prev) and re.match(r"^[a-zà-ỹđ]", t):
            parts[-1] = prev[:-1] + t
        else:
            parts.append(t)
    text = " ".join(parts)
    text = re.sub(r"\s+", " ", text).strip()
    # Make old spaced punctuation less noisy.
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?])(?=\S)", r"\1 ", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    return text


def reflow_lines_to_paragraphs(lines: list, page_rect) -> list:
    if not lines:
        return []
    lines = sorted(lines, key=lambda r: (r["bbox"][1], r["bbox"][0]))
    for r in lines:
        r["is_footnote"] = is_footnote_line(r, page_rect)
    med_h = median_line_height(lines)
    groups, cur = [], []
    for r in lines:
        if not cur:
            cur = [r]
        elif should_new_paragraph(cur[-1], r, med_h, page_rect):
            groups.append(cur)
            cur = [r]
        else:
            cur.append(r)
    if cur:
        groups.append(cur)
    out = []
    for i, g in enumerate(groups):
        text = join_paragraph_lines(g)
        if not text:
            continue
        b = [min(x["bbox"][0] for x in g), min(x["bbox"][1] for x in g), max(x["bbox"][2] for x in g), max(x["bbox"][3] for x in g)]
        if all(x.get("line_type") == "heading" for x in g) and len(text) <= 130:
            ptype = "heading"
        elif any(x.get("is_footnote") for x in g):
            ptype = "footnote"
        else:
            ptype = "body"
        q = text_metrics(text, lines=[x.get("text", "") for x in g])
        out.append({
            "paragraph_index_in_page": i, "paragraph_type": ptype, "text": text, "bbox": b,
            "line_count": len(g), "source": ";".join(sorted(set(x.get("source", "") for x in g))),
            "quality_score": q["quality_score"], "vietnamese_ratio": q["vietnamese_ratio"],
            "weird_char_ratio": q["weird_char_ratio"], "line_ids": ";".join(x.get("line_global_id", "") for x in g),
        })
    return out

ABBREV_PATTERNS = [
    "v.v.", "v.v..", "tr.C.N.", "T.P.", "P.", "S.", "HV.", "q.", "sđd.", "x.", "X.",
]


def protect_sentence_abbrevs(text: str) -> tuple:
    repl = {}
    protected = text
    # protect abbreviation dots
    for i, ab in enumerate(ABBREV_PATTERNS):
        key = f"§ABBR{i}§"
        protected = protected.replace(ab, key)
        repl[key] = ab
    # protect decimals and numbered references like A.69, HV.140.
    def repl_match(m):
        key = f"§DOT{len(repl)}§"
        repl[key] = m.group(0)
        return key
    protected = re.sub(r"\b[A-Z]{1,4}\.\d+\b", repl_match, protected)
    protected = re.sub(r"\b\d+\.\d+\b", repl_match, protected)
    return protected, repl


def unprotect_sentence_abbrevs(text: str, repl: dict) -> str:
    for k, v in repl.items():
        text = text.replace(k, v)
    return text


def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    text = norm_text(paragraph_text)
    if paragraph_type == "heading":
        return []
    if not text:
        return []
    protected, repl = protect_sentence_abbrevs(text)
    out = []
    buf = []
    depth = 0
    quote_open = False
    for i, ch in enumerate(protected):
        buf.append(ch)
        if ch in "([{" or ch == "«":
            depth += 1
        elif ch in ")]}" or ch == "»":
            depth = max(0, depth - 1)
        elif ch == '"':
            quote_open = not quote_open
        if ch in ".?!…;":
            tail = "".join(buf).strip()
            nxt = protected[i + 1:i + 8]
            next_char = protected[i + 1:i + 2]
            # Semicolon split only when sentence is already very long and next token looks like a new clause.
            if ch == ";" and len(tail) < 220:
                continue
            if depth > 0 and ch != ";":
                continue
            # Avoid splitting at numbered list/footnote markers or short fragments.
            if re.search(r"\(\s*\d+\s*\)$", tail):
                continue
            if next_char and not re.match(r"\s", next_char):
                continue
            # Find first non-space after punctuation.
            rest = protected[i + 1:]
            m = re.search(r"\S", rest)
            if m:
                nchar = rest[m.start()]
                if ch != ";" and not re.match(r"[A-ZÀ-ỸĐ0-9\(\"'«]", nchar):
                    continue
            sent = unprotect_sentence_abbrevs(tail, repl)
            sent = norm_text(sent)
            if sent:
                out.append(sent)
            buf = []
    remain = unprotect_sentence_abbrevs("".join(buf).strip(), repl)
    remain = norm_text(remain)
    if remain:
        # Merge tiny remainder into previous; historical prose often has fragments after OCR punctuation.
        if out and len(remain) < 18:
            out[-1] = norm_text(out[-1] + " " + remain)
        else:
            out.append(remain)
    # Final validation.
    final = []
    for s in out:
        q = text_metrics(s)
        if len(s) < 12 and q["domain_heading_hits"] == 0:
            continue
        if q["quality_score"] < MIN_KEEP_SENTENCE_QUALITY - 10 and q["word_count"] < 5:
            continue
        final.append(s)
    return final


In [ ]:

# ============================================================
# 9. Process one PDF and all PDFs
# ============================================================
all_final_lines = []
all_paragraphs = []
all_sentences = []
page_quality_report = []
source_selection_report = []
line_filter_audit = []
dropped_pages = []
dropped_lines = []
correction_log = []
suspicious_lines = []
suspicious_sentences = []


def make_work_id(pdf_path: Path, used: set) -> str:
    base = re.sub(r"[^A-Za-z0-9_\-]+", "_", pdf_path.stem).strip("_") or "pdf"
    wid = base
    k = 2
    while wid in used:
        wid = f"{base}_{k}"
        k += 1
    used.add(wid)
    return wid


def suspicious_reasons_for_text(text: str, q: dict) -> list:
    reasons = []
    if CLEAR_JUNK_RE.search(norm_text(text)):
        reasons.append("known_junk_pattern")
    if q["quality_score"] < MIN_KEEP_SENTENCE_QUALITY:
        reasons.append(f"low_quality:{q['quality_score']:.1f}")
    if q["weird_char_ratio"] > 0.035:
        reasons.append("weird_char_ratio")
    if q["word_count"] >= 8 and q["vietnamese_ratio"] < MIN_VIET_RATIO_FOR_LONG_TEXT:
        reasons.append("low_vietnamese_ratio")
    if q["repeated_char_ngram_ratio"] > 0.05:
        reasons.append("repeated_ngram_ratio")
    if q["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE:
        reasons.append("library_noise")
    if q["unaccented_vi_hits"] >= 3 and q["accent_ratio"] < 0.015:
        reasons.append("possible_missing_diacritics")
    return reasons


def process_pdf(pdf_path: Path, work_id: str, remaining_page_budget=None):
    print(f"\nProcessing {work_id}: {pdf_path}")
    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        dropped_pages.append({"work_id": work_id, "pdf_path": str(pdf_path), "page_number": None, "reason": f"cannot_open:{type(e).__name__}"})
        print("Cannot open PDF:", e)
        return 0

    total_pages = len(doc)
    content_start_page = estimate_content_start(doc, work_id, pdf_path)
    print("pages:", total_pages, "auto_content_start_page:", content_start_page)
    page_limit = total_pages if MAX_PAGES_PER_PDF is None else min(total_pages, MAX_PAGES_PER_PDF)
    if remaining_page_budget is not None:
        page_limit = min(page_limit, remaining_page_budget)
    reocr_state = {"used": 0}
    processed_pages = 0

    for page_idx in tqdm(range(page_limit), desc=work_id):
        processed_pages += 1
        page = doc[page_idx]
        page_number = page_idx + 1
        visual_m = visual_page_metrics(page)
        tl_lines_for_class = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
        tl_text_for_class = lines_to_text(tl_lines_for_class)
        tl_m_for_class = text_metrics(tl_text_for_class, lines=[r.get("raw_text", "") for r in tl_lines_for_class])
        page_class, class_reasons = classify_page(page_number, tl_text_for_class, tl_m_for_class, visual_m, content_start_page)

        # If text layer is too poor to classify but visual has nonblank text-like page, do a quick OCR for classification.
        classification_text = tl_text_for_class
        classification_m = tl_m_for_class
        if page_class == "junk_ocr_page" and tesseract_available() and not is_blank_page(tl_m_for_class, visual_m):
            quick_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=FAST_FRONTMATTER_OCR_DPI)
            quick_text = lines_to_text(quick_lines)
            quick_m = text_metrics(quick_text, lines=[r.get("raw_text", "") for r in quick_lines], ocr_conf_values=[r.get("ocr_conf") for r in quick_lines])
            if quick_m["quality_score"] > classification_m["quality_score"]:
                classification_text, classification_m = quick_text, quick_m
                page_class, class_reasons = classify_page(page_number, classification_text, classification_m, visual_m, content_start_page)

        report = {
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
            "content_start_page": content_start_page, "page_class": page_class,
            "class_reasons": ";".join(class_reasons) if class_reasons else "",
            **{f"visual_{k}": v for k, v in visual_m.items()},
            **{f"class_text_{k}": v for k, v in classification_m.items()},
        }

        if page_class in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark", "cover_title_front_matter", "front_matter_before_content"}:
            report.update({"selected_source": "dropped_by_page_classifier", "kept_lines": 0, "dropped_lines": 0, "final_sentences": 0})
            page_quality_report.append(report)
            dropped_pages.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "page_class": page_class, "drop_reason": ";".join(class_reasons) if class_reasons else page_class,
                **classification_m,
            })
            continue

        selected_lines, selected_source, selected_m, tl_m, ocr_m, source_reason = select_page_source(page, page_idx, work_id, pdf_path, page_class)
        source_selection_report.append({
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
            "page_class": page_class, "selected_source": selected_source, "source_reason": source_reason,
            **{f"text_layer_{k}": v for k, v in tl_m.items()},
            **{f"ocr_{k}": v for k, v in (ocr_m or {}).items()},
        })

        if not selected_lines or (DROP_LOW_CONF_OCR_PAGES and selected_m.get("quality_score", 0) < MIN_SELECTED_PAGE_QUALITY):
            report.update({"selected_source": selected_source, "kept_lines": 0, "dropped_lines": len(selected_lines), "final_sentences": 0})
            page_quality_report.append(report)
            dropped_pages.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "page_class": "junk_ocr_page" if selected_source == "drop_no_reliable_source" else page_class,
                "drop_reason": f"low_selected_source_quality:{selected_m.get('quality_score', 0):.1f};{source_reason}",
                **selected_m,
            })
            continue

        kept_lines = []
        dropped_count = 0
        seen_line_texts = Counter()
        for li, raw_line in enumerate(selected_lines):
            raw_line = dict(raw_line)
            raw_line["line_global_id"] = f"{work_id}_p{page_number:04d}_l{li+1:03d}"
            kept, is_kept, reasons = filter_line(raw_line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
            audit_row = {
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "line_id": raw_line["line_global_id"], "source": raw_line.get("source"), "page_class": page_class,
                "raw_text": raw_line.get("raw_text", ""), "kept": bool(is_kept), "reasons": ";".join(reasons),
                "bbox": json.dumps(raw_line.get("bbox", []), ensure_ascii=False), "ocr_conf": raw_line.get("ocr_conf"),
            }
            if is_kept and kept is not None:
                # Drop duplicate exact line repeated on same page, except headings.
                key = normalize_for_match(kept["text"])
                seen_line_texts[key] += 1
                if seen_line_texts[key] > 2 and kept.get("line_type") != "heading":
                    is_kept = False
                    reasons = reasons + ["duplicate_repeated_line"]
                    audit_row.update({"kept": False, "reasons": ";".join(reasons), "final_text": kept["text"]})
                else:
                    kept["page_class"] = page_class
                    kept["selected_source"] = selected_source
                    kept["source_reason"] = source_reason
                    kept_lines.append(kept)
                    audit_row.update({"final_text": kept["text"], "line_quality_score": kept.get("line_quality_score")})
                    sr = suspicious_reasons_for_text(kept["text"], line_quality(kept["text"], kept.get("ocr_conf")))
                    if sr:
                        suspicious_lines.append({**audit_row, "suspicious_reasons": ";".join(sr)})
            if not is_kept:
                dropped_count += 1
                dropped_lines.append({**audit_row, "kept": False, "reasons": ";".join(reasons)})
            line_filter_audit.append(audit_row)

        paragraphs = reflow_lines_to_paragraphs(kept_lines, page.rect)
        page_sentence_count = 0
        for kline in kept_lines:
            row = dict(kline)
            row["bbox"] = json.dumps(row.get("bbox", []), ensure_ascii=False)
            all_final_lines.append(row)

        for pi, para in enumerate(paragraphs):
            paragraph_id = f"{work_id}_p{page_number:04d}_para{pi+1:03d}"
            prow = {
                "paragraph_id": paragraph_id, "work_id": work_id, "pdf_path": str(pdf_path),
                "page_idx": page_idx, "page_number": page_number, **para,
                "bbox": json.dumps(para.get("bbox", []), ensure_ascii=False),
            }
            all_paragraphs.append(prow)
            if para["paragraph_type"] == "heading":
                continue
            sentences = split_sentences_vietnamese(para["text"], para["paragraph_type"])
            for si, sent in enumerate(sentences):
                sq = text_metrics(sent)
                s_reasons = suspicious_reasons_for_text(sent, sq)
                sent_id = f"{work_id}_s{len(all_sentences)+1:07d}"
                srow = {
                    "sent_id": sent_id, "paragraph_id": paragraph_id, "work_id": work_id,
                    "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                    "sentence_index_in_paragraph": si, "text": sent, "source": para.get("source"),
                    "paragraph_type": para.get("paragraph_type"), "bbox": prow["bbox"],
                    "quality_score": sq["quality_score"], "vietnamese_ratio": sq["vietnamese_ratio"],
                    "weird_char_ratio": sq["weird_char_ratio"], "dictionary_hit_ratio": sq["dictionary_hit_ratio"],
                    "suspicious_reasons": ";".join(s_reasons),
                }
                if s_reasons:
                    suspicious_sentences.append(srow)
                    # Keep suspicious if it is still above hard floor; otherwise drop from final sentence-only.
                    if sq["quality_score"] < MIN_KEEP_SENTENCE_QUALITY:
                        continue
                all_sentences.append(srow)
                page_sentence_count += 1

        report.update({
            "selected_source": selected_source, "source_reason": source_reason,
            **{f"selected_{k}": v for k, v in selected_m.items()},
            "kept_lines": len(kept_lines), "dropped_lines": dropped_count,
            "paragraphs": len(paragraphs), "final_sentences": page_sentence_count,
            "line_reocr_used_pdf": reocr_state.get("used", 0),
        })
        page_quality_report.append(report)

    doc.close()
    return processed_pages

used_ids = set()
page_budget = FAST_TEST_MAX_PAGES_TOTAL if FAST_TEST_MODE else None
start_time = time.time()
processed_pages_total = 0

for pdf_i, pdf_path in enumerate(pdf_paths, 1):
    if page_budget is not None and page_budget <= 0:
        break
    work_id = make_work_id(pdf_path, used_ids)
    n = process_pdf(pdf_path, work_id, remaining_page_budget=page_budget)
    processed_pages_total += n
    if page_budget is not None:
        page_budget -= n

elapsed = time.time() - start_time
print("\nDone processing")
print("processed_pages:", processed_pages_total)
print("final_lines:", len(all_final_lines))
print("paragraphs:", len(all_paragraphs))
print("sentences:", len(all_sentences))
print("dropped_pages:", len(dropped_pages))
print("dropped_lines:", len(dropped_lines))
print("elapsed_min:", round(elapsed / 60, 2))


In [ ]:

# ============================================================
# 10. Export final files + audit + zip
# ============================================================
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PKG_DIR.mkdir(parents=True, exist_ok=True)

lines_df = pd.DataFrame(all_final_lines)
paras_df = pd.DataFrame(all_paragraphs)
sents_df = pd.DataFrame(all_sentences)
page_df = pd.DataFrame(page_quality_report)
source_df = pd.DataFrame(source_selection_report)
line_audit_df = pd.DataFrame(line_filter_audit)
dropped_pages_df = pd.DataFrame(dropped_pages)
dropped_lines_df = pd.DataFrame(dropped_lines)
correction_df = pd.DataFrame(correction_log)
susp_lines_df = pd.DataFrame(suspicious_lines)
susp_sents_df = pd.DataFrame(suspicious_sentences)

# Stable sort.
for df, cols in [
    (lines_df, ["work_id", "page_number", "line_global_id"]),
    (paras_df, ["work_id", "page_number", "paragraph_id"]),
    (sents_df, ["work_id", "page_number", "sent_id"]),
]:
    if not df.empty:
        use_cols = [c for c in cols if c in df.columns]
        df.sort_values(use_cols, inplace=True)
        df.reset_index(drop=True, inplace=True)

final_lines_csv = FINAL_DIR / "final_lines.csv"
final_paras_csv = FINAL_DIR / "final_paragraphs.csv"
final_sents_csv = FINAL_DIR / "final_sentences_only.csv"
final_sents_auto_csv = FINAL_DIR / "final_sentences_auto.csv"  # compatibility copy; sentence-only in this notebook.

page_report_csv = AUDIT_DIR / "page_quality_report.csv"
line_filter_csv = AUDIT_DIR / "line_filter_audit.csv"
correction_csv = AUDIT_DIR / "correction_log.csv"
susp_lines_csv = AUDIT_DIR / "suspicious_lines.csv"
susp_sents_csv = AUDIT_DIR / "suspicious_sentences.csv"
dropped_pages_csv = AUDIT_DIR / "dropped_pages.csv"
dropped_lines_csv = AUDIT_DIR / "dropped_lines.csv"
source_selection_csv = AUDIT_DIR / "source_selection_report.csv"
summary_by_work_csv = AUDIT_DIR / "summary_quality_by_work_id.csv"
summary_csv = AUDIT_DIR / "run_summary.csv"

lines_df.to_csv(final_lines_csv, index=False, encoding="utf-8-sig")
paras_df.to_csv(final_paras_csv, index=False, encoding="utf-8-sig")
sents_df.to_csv(final_sents_csv, index=False, encoding="utf-8-sig")
sents_df.to_csv(final_sents_auto_csv, index=False, encoding="utf-8-sig")
page_df.to_csv(page_report_csv, index=False, encoding="utf-8-sig")
line_audit_df.to_csv(line_filter_csv, index=False, encoding="utf-8-sig")
correction_df.to_csv(correction_csv, index=False, encoding="utf-8-sig")
susp_lines_df.to_csv(susp_lines_csv, index=False, encoding="utf-8-sig")
susp_sents_df.to_csv(susp_sents_csv, index=False, encoding="utf-8-sig")
dropped_pages_df.to_csv(dropped_pages_csv, index=False, encoding="utf-8-sig")
dropped_lines_df.to_csv(dropped_lines_csv, index=False, encoding="utf-8-sig")
source_df.to_csv(source_selection_csv, index=False, encoding="utf-8-sig")

# Per-PDF final text from sentence-only final, with page separators.
if not sents_df.empty:
    for work_id, g in sents_df.groupby("work_id", dropna=False):
        parts = []
        cur_page = None
        for _, r in g.iterrows():
            if r.get("page_number") != cur_page:
                cur_page = r.get("page_number")
                parts.append(f"\n\n[Page {cur_page}]\n")
            parts.append(norm_text(r.get("text", "")))
        (TEXT_DIR / f"{work_id}_final.txt").write_text("\n".join(x for x in parts if norm_text(x)), encoding="utf-8")

# Summary by work_id.
if not page_df.empty:
    pages_summary = page_df.groupby("work_id", dropna=False).agg(
        total_pages=("page_number", "count"),
        kept_pages=("kept_lines", lambda x: int(pd.to_numeric(x, errors="coerce").fillna(0).gt(0).sum())),
        dropped_pages=("selected_source", lambda x: int(x.astype(str).str.contains("dropped|drop", regex=True).sum())),
        avg_selected_quality=("selected_quality_score", "mean") if "selected_quality_score" in page_df.columns else ("class_text_quality_score", "mean"),
        final_sentences=("final_sentences", "sum"),
        dropped_lines=("dropped_lines", "sum"),
    ).reset_index()
else:
    pages_summary = pd.DataFrame()

if not sents_df.empty:
    sent_summary = sents_df.groupby("work_id", dropna=False).agg(
        final_sentence_rows=("sent_id", "count"),
        suspicious_sentence_rows=("suspicious_reasons", lambda x: int(x.astype(str).str.len().gt(0).sum())),
        avg_sentence_quality=("quality_score", "mean"),
        low_vi_ratio_sentences=("vietnamese_ratio", lambda x: int(pd.to_numeric(x, errors="coerce").fillna(0).lt(MIN_VIET_RATIO_FOR_LONG_TEXT).sum())),
    ).reset_index()
    if not pages_summary.empty:
        summary_by_work = pages_summary.merge(sent_summary, on="work_id", how="outer")
    else:
        summary_by_work = sent_summary
else:
    summary_by_work = pages_summary

if not summary_by_work.empty:
    # A rough 0-100 score useful for ranking audit priority, not a guarantee of correctness.
    summary_by_work["quality_score_by_work_id"] = (
        pd.to_numeric(summary_by_work.get("avg_sentence_quality", 0), errors="coerce").fillna(0).clip(0, 100) * 0.55 +
        pd.to_numeric(summary_by_work.get("avg_selected_quality", 0), errors="coerce").fillna(0).clip(0, 100) * 0.45
    ).round(2)
summary_by_work.to_csv(summary_by_work_csv, index=False, encoding="utf-8-sig")

run_summary = pd.DataFrame([{
    "pdf_count": len(pdf_paths),
    "total_pages": int(len(page_df)) if not page_df.empty else processed_pages_total,
    "kept_pages": int(pd.to_numeric(page_df.get("kept_lines", pd.Series(dtype=float)), errors="coerce").fillna(0).gt(0).sum()) if not page_df.empty else 0,
    "dropped_pages": int(len(dropped_pages_df)),
    "total_lines": int(len(lines_df) + len(dropped_lines_df)),
    "final_lines": int(len(lines_df)),
    "dropped_lines": int(len(dropped_lines_df)),
    "final_paragraphs": int(len(paras_df)),
    "final_sentences": int(len(sents_df)),
    "suspicious_lines": int(len(susp_lines_df)),
    "suspicious_sentences": int(len(susp_sents_df)),
    "correction_events": int(len(correction_df)),
    "tesseract_available": tesseract_available(),
    "fast_test_mode": FAST_TEST_MODE,
    "output_dir": str(OUTPUT_DIR),
}])
run_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

# Package output.
zip_path = PKG_DIR / "dntc_auto_output.zip"
if ZIP_OUTPUT:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in [
            final_lines_csv, final_paras_csv, final_sents_csv, final_sents_auto_csv,
            page_report_csv, line_filter_csv, correction_csv, susp_lines_csv, susp_sents_csv,
            dropped_pages_csv, dropped_lines_csv, source_selection_csv, summary_by_work_csv, summary_csv,
        ]:
            if Path(p).exists():
                zf.write(p, arcname=str(Path(p).relative_to(OUTPUT_DIR)))
        for p in TEXT_DIR.glob("*.txt"):
            zf.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))

print("\n===== RUN SUMMARY =====")
print(run_summary.to_string(index=False))
if not summary_by_work.empty:
    print("\n===== QUALITY BY WORK_ID =====")
    display(summary_by_work.sort_values("quality_score_by_work_id", ascending=True).head(30))
print("\nFinal lines:", final_lines_csv)
print("Final paragraphs:", final_paras_csv)
print("Final sentences only:", final_sents_csv)
print("Compatibility final_sentences_auto:", final_sents_auto_csv)
print("Audit dir:", AUDIT_DIR)
print("output_zip_path:", zip_path if ZIP_OUTPUT else "ZIP_OUTPUT=False")


In [ ]:

# ============================================================
# 11. Acceptance sanity checks and examples
# ============================================================
BAD_EXAMPLE_PATTERNS = [
    "OPOCerererore", "Fel Fat Sek", "Seer tit", "mADS ee", ": wa", ". ‘ y", "PHÀM",
    "Đầu thé ky", "doi #ự Đức", "Nha Tuy", "Nước tả", "chi€m", "bi€n", "huyénhién", "Cao Mén",
]

if not sents_df.empty:
    print("final_sentences_only rows:", len(sents_df))
    # final_sentences_only must not contain paragraph/heading rows. It has no type column by design; paragraph_type can be body/footnote.
    print("paragraph_type distribution in sentence-only final:")
    print(sents_df.get("paragraph_type", pd.Series(dtype=str)).value_counts(dropna=False).to_string())
    print("\nBad pattern search in final_sentences_only:")
    for pat in BAD_EXAMPLE_PATTERNS:
        cnt = int(sents_df["text"].astype(str).str.contains(re.escape(pat), case=False, regex=True, na=False).sum())
        print(f"{pat}: {cnt}")
    print("\nTop suspicious sentences audit examples:")
    if not susp_sents_df.empty:
        display(susp_sents_df[[c for c in ["work_id", "page_number", "quality_score", "suspicious_reasons", "text"] if c in susp_sents_df.columns]].head(30))
    else:
        print("No suspicious sentences by current heuristic.")
else:
    print("No final sentences produced. Check dropped_pages.csv and source_selection_report.csv.")

if not dropped_pages_df.empty:
    print("\nDropped pages by class:")
    print(dropped_pages_df.get("page_class", pd.Series(dtype=str)).value_counts(dropna=False).to_string())

if not line_audit_df.empty:
    print("\nLine filter kept/drop counts:")
    print(line_audit_df["kept"].value_counts(dropna=False).to_string())
